In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from data_tools.db import Manager
from scipy.stats import loguniform, randint, uniform
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from xgboost import XGBRegressor

engine_fadn = Manager("fadn-191", "fadn").engine
engine_mvh = Manager("mvh-admin", "mvh").engine

In [ ]:
REFRESH_FADN_ML = False
FADN_ML_PATH = Path("input/fadn_ml.parquet")

if FADN_ML_PATH.exists() and not REFRESH_FADN_ML:
    tbl_fadn_ml = pd.read_parquet(FADN_ML_PATH)
else:
    sql = Path("sql/fadn_ml.sql").read_text(encoding="utf-8")
    tbl_fadn_ml = pd.read_sql(sql, con=engine_fadn)

    FADN_ML_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tbl_fadn_ml.to_parquet(
        FADN_ML_PATH,
        index=False,
    )

In [ ]:
tbl_fadn_ml["eme"] = (
    tbl_fadn_ml["tarsas_munkaora"] + tbl_fadn_ml["egyeni_munkaora"]
) / 1800

In [ ]:
tbl_2024 = tbl_fadn_ml[tbl_fadn_ml["ev"].eq(2024)].copy()
tbl_learn = tbl_2024[["eme", "legal", "osszes_mgi_ter_ha"]]

In [ ]:
# hasznositasi_kodtar_2026 = pd.read_excel(
#     "input/hasznositasi_kodtar_2026.xlsx", engine="calamine"
# )

In [ ]:
SQL = """
WITH tbl_ugyfela AS (
    SELECT
        regszam,
        CASE
            WHEN vallalkozasi_forma IS NULL
                 OR vallalkozasi_forma IN (
                     'Adószámmal rendelkező magánszemély',
                     'Egyéni vállalk.',
                     'Egyéb önálló vállalkozó'
                 )
            THEN 1
            ELSE 2
        END AS legal
    FROM altalanos.ugyfela
    WHERE ev = 2026
),

tbl_tera AS (
    SELECT
        regszam,
        SUM(terulet_elf) AS osszes_mgi_ter_ha
    FROM ek_adatok.tera
    WHERE ev = 2026 and terulet_elf > 0
    GROUP BY regszam
),

cis_allat AS (
    SELECT
        regszam,

        COALESCE(
            SUM(letszam_ig) FILTER (
                WHERE allat_tip = 'anyajuh'
            ),
            0
        ) AS anyajuh_atlag_allomany_db,

        COALESCE(
            SUM(letszam_ig) FILTER (
                WHERE allat_tip = 'anyatehén'
            ),
            0
        ) AS anyatehen_atlag_allomany_db,

        COALESCE(
            SUM(letszam_ig) FILTER (
                WHERE allat_tip = 'hízottbika'
            ),
            0
        ) AS hizo_bika_atlag_allomany_db,

        COALESCE(
            SUM(letszam_ig) FILTER (
                WHERE allat_tip = 'tejhasznú tehén'
            ),
            0
        ) AS tejelo_tehen_atlag_allomany_db

    FROM mvh_tk_kozv.tam_allat
    WHERE ev = 2026
      AND allat_tip IN (
          'anyajuh',
          'anyatehén',
          'hízottbika',
          'tejhasznú tehén'
      )
    GROUP BY regszam
)

SELECT
    t.regszam,
    t.osszes_mgi_ter_ha,
    u.legal,

    COALESCE(a.anyajuh_atlag_allomany_db, 0)
        AS anyajuh_atlag_allomany_db,

    COALESCE(a.anyatehen_atlag_allomany_db, 0)
        AS anyatehen_atlag_allomany_db,

    COALESCE(a.hizo_bika_atlag_allomany_db, 0)
        AS hizo_bika_atlag_allomany_db,

    COALESCE(a.tejelo_tehen_atlag_allomany_db, 0)
        AS tejelo_tehen_atlag_allomany_db

FROM tbl_tera AS t

LEFT JOIN tbl_ugyfela AS u
    USING (regszam)

LEFT JOIN cis_allat AS a
    USING (regszam);
"""

base_data = pd.read_sql(SQL, con=engine_mvh)
base_data.to_parquet("input/base_data_ml.parquet")

In [ ]:
tbl_2024.columns

In [ ]:
# ============================================================
# Beállítások
# ============================================================

FEATURES = [
    "legal",
    "osszes_mgi_ter_ha",
    "tejelo_tehen_atlag_allomany_db",
    "anyajuh_atlag_allomany_db",
    "hizo_bika_atlag_allomany_db",
]

TARGET = "eme"

# Rétegzéshez használt ÉME-méretkategóriák
SIZE_BINS = [-np.inf, 0.1, 1, 5, 15, 50, np.inf]


# ============================================================
# Tanítóadat
# ============================================================

train = (
    tbl_2024[FEATURES + [TARGET]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .query("eme >= 0")
    .copy()
)

X = train[FEATURES]
y = train[TARGET]


# ============================================================
# Méretkategóriák a rétegzéshez
# ============================================================

size_group = pd.cut(
    y,
    bins=SIZE_BINS,
    labels=False,
    include_lowest=True,
)

print("ÉME-méretkategóriák:")
print(
    pd.cut(
        y,
        bins=SIZE_BINS,
        include_lowest=True,
    )
    .value_counts()
    .sort_index()
)


# ============================================================
# Train / test split – ÉME-méret szerint rétegezve
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=size_group,
)


# ============================================================
# XGBoost + log target
# ============================================================

xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,
)

model = TransformedTargetRegressor(
    regressor=xgb,
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,
)


# ============================================================
# Hyperparaméter-tér
# ============================================================

param_distributions = {
    # Modell mérete
    "regressor__n_estimators": randint(200, 1200),
    # Fák komplexitása
    "regressor__max_depth": randint(2, 9),
    "regressor__min_child_weight": randint(1, 15),
    # Tanulási ráta
    "regressor__learning_rate": loguniform(0.01, 0.15),
    # Sor- és oszlopmintavétel
    "regressor__subsample": uniform(0.6, 0.4),
    "regressor__colsample_bytree": uniform(0.5, 0.5),
    # Regularizáció
    "regressor__gamma": uniform(0.0, 1.0),
    "regressor__reg_alpha": loguniform(1e-4, 10),
    "regressor__reg_lambda": loguniform(0.1, 20),
}


# ============================================================
# Cross-validation – ÉME-méret szerint rétegezve
# ============================================================

size_group_train = pd.cut(
    y_train,
    bins=SIZE_BINS,
    labels=False,
    include_lowest=True,
)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

cv = list(
    skf.split(
        X_train,
        size_group_train,
    )
)


search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=100,
    scoring="neg_mean_absolute_error",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True,
)


# ============================================================
# Hyperparaméter-hangolás
# ============================================================

search.fit(
    X_train,
    y_train,
)

print("\nBest parameters:")
for key, value in search.best_params_.items():
    print(f"{key}: {value}")

print(f"\nCV MAE: {-search.best_score_:.3f} ÉME")


# ============================================================
# Holdout teszt
# ============================================================

best_model = search.best_estimator_

y_pred = best_model.predict(X_test)
y_pred = np.clip(y_pred, 0, None)

print("\nHoldout performance")
print(f"MAE:    {mean_absolute_error(y_test, y_pred):.3f}")
print(f"MedAE:  {median_absolute_error(y_test, y_pred):.3f}")
print(f"RMSE:   {mean_squared_error(y_test, y_pred) ** 0.5:.3f}")
print(f"R²:     {r2_score(y_test, y_pred):.3f}")


# ============================================================
# Aggregált torzítás
# ============================================================

aggregate_bias = (y_pred.sum() - y_test.sum()) / y_test.sum()

print(f"Aggregate bias: {aggregate_bias:.1%}")


# ============================================================
# Hibaeloszlás
# ============================================================

abs_error = (y_test - y_pred).abs()

print(abs_error.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))


# ============================================================
# Teljesítmény ÉME-méret szerint
# ============================================================

result = pd.DataFrame(
    {
        "actual": y_test,
        "predicted": y_pred,
    }
)

result["error"] = result["predicted"] - result["actual"]

result["abs_error"] = result["error"].abs()

result["group"] = pd.cut(
    result["actual"],
    bins=SIZE_BINS,
    include_lowest=True,
)

size_summary = result.groupby("group", observed=True).agg(
    n=("actual", "size"),
    actual_mean=("actual", "mean"),
    predicted_mean=("predicted", "mean"),
    bias=("error", "mean"),
    mae=("abs_error", "mean"),
    medae=("abs_error", "median"),
)

display(size_summary)


# ============================================================
# Train / test méreteloszlás ellenőrzése
# ============================================================

split_summary = pd.concat(
    {
        "train": (
            pd.cut(
                y_train,
                bins=SIZE_BINS,
                include_lowest=True,
            )
            .value_counts(normalize=True)
            .sort_index()
        ),
        "test": (
            pd.cut(
                y_test,
                bins=SIZE_BINS,
                include_lowest=True,
            )
            .value_counts(normalize=True)
            .sort_index()
        ),
    },
    axis=1,
)

display(split_summary)


# ============================================================
# Ellenőrzés legal szerint
# ============================================================

validation = X_test.copy()

validation["eme"] = y_test
validation["eme_est"] = y_pred

validation_summary = validation.groupby("legal").agg(
    n=("eme", "size"),
    eme_mean=("eme", "mean"),
    eme_est_mean=("eme_est", "mean"),
    eme_median=("eme", "median"),
    eme_est_median=("eme_est", "median"),
)

display(validation_summary)


# ============================================================
# Végleges modell újratanítása az összes tanítóadaton
# ============================================================

best_model.fit(
    X,
    y,
)


# ============================================================
# Becslés a teljes populációra
# ============================================================

tbl_population = base_data[FEATURES].copy()

tbl_population["eme_est"] = np.nan

valid_mask = (
    tbl_population[FEATURES].replace([np.inf, -np.inf], np.nan).notna().all(axis=1)
)

tbl_population.loc[valid_mask, "eme_est"] = best_model.predict(
    tbl_population.loc[
        valid_mask,
        FEATURES,
    ]
)

tbl_population["eme_est"] = tbl_population["eme_est"].clip(lower=0)


# ============================================================
# Visszaírás base_data-ba
# ============================================================

base_data["eme_est"] = tbl_population["eme_est"]


# ============================================================
# Ellenőrzés
# ============================================================

display(base_data[FEATURES + ["eme_est"]].head())

print()
print(base_data["eme_est"].describe())

print(f"\nBecsült összes ÉME: {base_data['eme_est'].sum():,.0f}".replace(",", " "))